# 1D J1J2 Inference: RNN (Euclidean, Poincare, Lorentz)

This notebook is part of the work arXiv:2604.24337 [quant-ph]. In this notebook, I performed the inference process for the trained neural quantum states using 10000 samples. The loading directory in the Github repo might differ from the loading path used here. Please check and use the correct loading path. 

The kernel as to be restarted in order to run cells involving NQS with different code settings. In particular:

- LorentzRNN (J2=0.0, 0.2): double clamps inside 'forward' method on both `state` and `new_h`. Default `spatial_norm=20` (used inside the `project_lorentz_manual` function) in both the files j1j2_definitions_manifold_update.py and j1j2_wf_lorentz.py. 
- Lorentz RNN (J2=0.5, 0.8): single-clamp (inside the `forward` method) - need to comment out the norm clamp on `new_h` in the `forward` method of the LorentzRNN class in the file j1j2_definitions_manifold_update.py single clamp (need to modify the forward pass to get rid of the clamp on new_h). Change the default to `spatial_norm=18` (used inside the `project_lorentz_manual` function) in both the files j1j2_definitions_manifold_update.py and j1j2_wf_lorentz.py.

- PoincareRNN: J2=0.0,0.2,0.5: Need to comment out the norm_clamps:
`#W_otimes_h = self.norm_clamp(W_otimes_h)`   
`#U_otimes_x = self.norm_clamp(U_otimes_x)`
inside the `one_rnn_transform` method in j1j2_poincare_definitions.py
- PoincareRNN: J2=0.8: Uncomment the above lines in `one_rnn_transform` method in j1j2_poincare_definitions.py


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import time
sys.path.append('utility_lorentz')
from j1j2_train_loop_lorentz import *

sys.path.append('utility_poincare')
from j1j2_hyprnn_train_loop import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False
GPU Available:  False


In [2]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))    
    if mad == 0:
        return eloc       
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad    
    # Clip the values (keeping the imaginary part if it exists)
    # Create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)   
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test(wf, numsamples, path_to_weights, Ee, clipped_e = False):
    if 'Lorentz' in wf.name:
        test_samples_before = wf.sample_no_tau(numsamples)
    else: 
        test_samples_before = wf.sample(numsamples)
    print(f'The number of samples is {len(test_samples_before)}')
    # --- PART A: Check performance BEFORE loading (Baseline) ---
    wf.model.eval() 
    with torch.no_grad():
        test_gs_before = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_before, Marshall_sign=True)
        gs_mean_b = np.round(np.mean(test_gs_before),4)
        gs_var_b = np.round(np.var(test_gs_before),4)
    print(f'Before loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_b}, var E = {gs_var_b}')
    print('====================================================================')
     # --- PART B: Remap and Load the Weights ---
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    #  Load the re_mapped weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")   
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        if 'Lorentz' in wf.name:
            test_samples_after = wf.sample_no_tau(numsamples)
        else: 
            test_samples_after = wf.sample(numsamples)               
        if clipped_e:
            # Get raw energies
            raw_gs_after = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)  
            # Clipping
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)   
            # Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)    
            # Count how many were clipped 
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [3]:
N=100
numsamples = 10000
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fcn = 'trained_nqs/J1J2'

## J2=0.0

In [5]:
J2_ = 0.0
J2 = +J2_ * np.ones(syssize)
Ee_00=-44.12773986967251

In [10]:
# EuclRNN: -25.3726
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/RNN_euclidean/J2={J2_}/N100_J1=1.0|J2={J2_}_EuclRNN_70_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (24.774799346923828-0j), var E = 0.1753000020980835
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-25.372600555419922+0.0003000000142492354j), var E = 0.03629999980330467
DMRG energy  is -44.1277
Time taken=0.024 hrs


In [6]:
# PoincareRNN: -42.9945
#in one_rnn_transform, comment out the norm_clamps:
#W_otimes_h = self.norm_clamp(W_otimes_h)   
#U_otimes_x = self.norm_clamp(U_otimes_x)
#otherwise, wrong result: Mean E = (-24.708999633789062-0.03709999844431877j), var E = 29.691499710083008
rmax=0.618
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/RNN_poincare/J2=0.0_rmax=0.618/N100_J1=1.0|J2=0.0_HypRNN_70_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (24.774799346923828-0j), var E = 0.1753000020980835
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.99449920654297-0.0006000000284984708j), var E = 0.4747999906539917
DMRG energy  is -44.1277
Time taken=0.119 hrs


In [15]:
#LorentzRNN: spatial_clamp = 4.0, double clamp: -44.0163
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=4.0, seed=111)
h_wl = f'{fcn}/RNN_lorentz/J2=0.0_double_sc=4.0/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_{units}_ns=80_MsTrue_checkpoint.pt'
# The last parameter in LorentzRNN is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (24.760400772094727-9.999999747378752e-05j), var E = 0.15729999542236328
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-44.016300201416016+0.0010000000474974513j), var E = 0.48429998755455017
DMRG energy  is -44.1277
Time taken=0.196 hrs


## J2=0.2

In [7]:
J2_ = 0.2
J2 = +J2_ * np.ones(syssize)
Ee_02=-40.7388

In [17]:
# EuclRNN: -6.0552
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/RNN_euclidean/J2={J2_}/N100_J1=1.0|J2={J2_}_EuclRNN_70_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (29.673799514770508-0j), var E = 0.272599995136261
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-6.055200099945068-0.0013000000035390258j), var E = 0.04740000143647194
DMRG energy  is -40.7388
Time taken=0.036 hrs


In [8]:
# PoincareRNN: -39.8376
#(must comment out norm_clamps in one_rnn_transform, similar to above)
#otherwise, wrong result: Mean E = (-20.03420066833496+0.01979999989271164j), var E = 28.87459945678711
rmax=0.618
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/RNN_poincare/J2=0.2_rmax=0.618/N100_J1=1.0|J2=0.2_HypRNN_70_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (29.673799514770508-0j), var E = 0.272599995136261
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.83760070800781-0.002300000051036477j), var E = 0.48159998655319214
DMRG energy  is -40.7388
Time taken=0.211 hrs


In [20]:
#LorentzRNN: -40.528, spatial_clamp = 4.0, double clamp
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=4.0, seed=111)
h_wl = f'{fcn}/RNN_lorentz/J2=0.2_double_sc=4.0/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_{units}_ns=80_MsTrue_checkpoint.pt'
# The last parameter in LorentzRNN is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (29.65169906616211-0.00019999999494757503j), var E = 0.2361000031232834
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.52799987792969-0.002899999963119626j), var E = 0.4142000079154968
DMRG energy  is -40.7388
Time taken=0.372 hrs


## J2=0.5

In [4]:
J2_ = 0.5
J2 = +J2_ * np.ones(syssize)
Ee_05=37.5000

In [22]:
# EuclRNN: -12.8900
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/RNN_euclidean/J2={J2_}/N100_J1=1.0|J2={J2_}_EuclRNN_70_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (37.02220153808594-0j), var E = 0.48260000348091125
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-12.890000343322754+0.0010000000474974513j), var E = 0.20239999890327454
DMRG energy  is 37.5
Time taken=0.048 hrs


In [10]:
# PoincareRNN: -35.8290 (comment out norm_clamps in one_rnn_transform)
# old name: N100_J1=1.0|J2=0.5_HypRNN_70_id_hyp_ns=80_lr2=5e-3_m36_MsTrue_meanE.npy
rmax=0.618
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/RNN_poincare/J2=0.5_rmax=0.618/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (37.02220153808594-0j), var E = 0.48260000348091125
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-35.82899856567383-0.0027000000700354576j), var E = 1.029099941253662
DMRG energy  is 37.5
Time taken=0.214 hrs


In [5]:
#LorentzRNN: spatial_clamp = 2.0, single clamp (need to modify the forward pass to get rid of the clamp on new_h)
# spatial norm (inside the 'project_lorentz_manual' function) = 18. Default is 20 
#change  'norm_spatial' in j1j2_definitions_manifold_update and j1j2_wf_lorentz
# if the default spatial_norm=20 is used, wrong result below:
#Mean E = (7.613900184631348-0.007499999832361937j), var E = 29.70949935913086
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=2.0, seed=111)
h_wl = f'{fcn}/RNN_lorentz/J2=0.5_single_sc=2.0/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_{units}_ns=80_MsTrue_checkpoint.pt'
# The last parameter in LorentzRNN is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.988800048828125-0j), var E = 0.4198000133037567
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.0181999206543-0.003000000026077032j), var E = 0.28630000352859497
DMRG energy  is 37.5
Time taken=0.56 hrs


## J2=0.8

In [6]:
J2_ = 0.8
J2 = +J2_ * np.ones(syssize)
Ee_08=-42.07006297371643

In [28]:
# EuclRNN: -5.8802
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/RNN_euclidean/J2={J2_}/N100_J1=1.0|J2={J2_}_EuclRNN_70_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (44.37070083618164+0j), var E = 0.7695000171661377
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-5.880199909210205-0.000699999975040555j), var E = 0.02019999921321869
DMRG energy  is -42.0701
Time taken=0.049 hrs


In [7]:
#Poincare:  with double clamp in the forward pass (by default), 
# Need to uncomment (put back) the norm_clamps in one_rnn_transform - restart kernel to run this cell. 
# otherwise the result will be wrong, e.g: Mean E = (-24.85580062866211+0.017400000244379044j), var E = 10.793100357055664
# above result obtained with no clamp on 'new_h'
rmax=0.7
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/RNN_poincare/J2=0.8_rmax=0.7/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax=0.7_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (44.37070083618164+0j), var E = 0.7695000171661377
Successfully remapped and loaded weights.
Clipped 319 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-35.52920150756836+0.005200000014156103j), var E = 9.200300216674805
DMRG energy  is -42.0701
Time taken=0.23 hrs


In [8]:
#LorentzRNN: spatial_clamp = 2.0, single clamp (need to modify the forward pass to get rid of the clamp on new_h)
# spatial norm=18 (used inside the 'project_lorentz_manual' function) - defined in j1j2_definitions... and j1j2_wf_lorentz
# if spatial_norm is kept at the default value 20: wrong result below
#Mean E = (-6.560500144958496+0.029999999329447746j), var E = 49.26029968261719
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=2.0, seed=111)
h_wl = f'{fcn}/RNN_lorentz/J2=0.8_single_sc=2.0/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_{units}_ns=80_MsTrue_checkpoint.pt'
# The last parameter in LorentzRNN is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (44.325801849365234-0.0003000000142492354j), var E = 0.6710000038146973
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.35459899902344-0.00139999995008111j), var E = 0.47290000319480896
DMRG energy  is -42.0701
Time taken=0.511 hrs
